## 03 - Testing the Paper's Reference-Processing Procedure (n=150, k=9, Paper's Gene List)

### Aim
* Test whether matching the paper's own single-cell-reference processing - pseudo-bulking (sum of 5
  coverage-ranked cells), depth-matching, and its bounded Equation 1 transform - closes the remaining gap
  between this project's XDec reproduction and XDec-CHI's reported TCGA k=9 result, on top of the cohort
  (n=150) and gene-list (paper's 282 genes) fixes already tested in `01_restrict_to_panCanAtlas_cohort_
  k9.ipynb` (Run A).

### Input
* `data/raw/zenodo_3969339/StdWf1_PRJCA001063_annotated.h5ad` - full Peng et al. single-cell reference
  (Besca-reprocessed), re-read here to draw a larger per-class subsample than notebook 04's original export.
* `data/deconvolution/xdec/TCGA-PAAD_counts_raw_n150.tsv` - n=150 restricted bulk counts (notebook 01).
* `data/deconvolution/xdec/gene_lists/xdec_chi_table_s3_genes.txt` - the paper's 282-gene Table S3 list.

### Output
* `data/deconvolution/xdec/paper_style_ref/{refs_500.tsv, ref_metadata_500.tsv}` - 500-cells-per-class
  (vs. notebook 04's 300-cell cap) re-export, needed to reach up to 100 pseudo-bulks/type.
* `data/deconvolution/xdec/paper_style_ref/{refs_paperstyle.tsv, ref_metadata_paperstyle.tsv}` - **Run C**:
  the manually Equation-1-transformed pseudo-bulk reference - later found to be double-transformed (Step 6).
* `data/deconvolution/xdec/paper_style_ref/{refs_pseudobulk_raw.tsv, ref_metadata_pseudobulk_raw.tsv}` -
  **Run D**: the corrected, untransformed pseudo-bulk reference (`xdec.py` normalizes it itself).
* `data/deconvolution/xdec/{stage1,stage2}_with_refs_k9_n150_paperGenes_paperRef/` - Run C (flawed).
* `data/deconvolution/xdec/{stage1,stage2}_with_refs_k9_n150_paperGenes_pseudobulkRaw/` - Run D (corrected).

### Steps
1. Document the paper's exact reference-processing procedure (STAR Methods) and a data-availability
   limitation discovered while implementing it.
2. Re-export up to 500 single cells/class from the raw h5ad (notebook 04's original export capped at 300,
   too few to reach 100 pseudo-bulks/type for the 9 major classes).
3. Build pseudo-bulks: group cells per type, sum every 5, cap at 100 pseudo-bulks/type.
4. Depth-match pseudo-bulks to the highest-depth one; apply the paper's Equation 1 transform
   (`y = 1 - exp(-a*x), a = 1/max(x)`) per gene across all pseudo-bulks - **Run C**.
5. Rerun XDec Stage 1 (paper's gene list, Run C's reference, n=150, k=9) and Stage 2; apply the same three
   artifact signals used throughout this project (prevalence, correlation breadth, zero-floor DE genes).
   Compare against baseline (344 genes), Run A (paper genes), and Run B (no housekeeping).
6. Discover that `xdec.py` already implements joint quantile normalization + Equation 1 automatically
   (it is a port of the paper's own R codebase) - meaning Run C's manual pre-transform double-applied
   Equation 1 and corrupted the joint normalization. Rebuild the pseudo-bulk reference as **raw**,
   untransformed counts (**Run D**) and rerun, letting `xdec.py`'s own pipeline do all normalization.

### Result
* **Run C (flawed)**: no improvement over the gene-list-only fixes (breadth 6/10) - but this result was
  confounded by a real implementation bug, not a clean test (Step 6).
* **Run D (corrected)**: still no improvement - if anything, both recurring patterns got *more* extreme.
  Broadest-correlating profile breadth rose to 7/10 (worse than Runs A/B/C); the separate dominant-majority,
  zero-correlation profile became dominant in **102/150 samples (68%)** - the largest fraction seen in any
  run tested across this entire investigation (prior range: 82-88/150, ~55-59%), still correlating with
  *none* of the 10 reference types.

### Conclusion
* **The reference-processing hypothesis is decisively not confirmed**, with a methodologically sound
  (bug-fixed) test: neither single-cell-vs-pseudo-bulk reference construction nor the specific normalization
  details explain the artifact or the dominant-majority profile. Across every configuration tested in this
  project (cohort size, gene list, reference processing, individually and combined), the dominant-majority/
  zero-correlation profile not only persists but grows - the single strongest, most consistent finding of the
  whole investigation, and the strongest evidence yet that it represents real tumor-epithelial biology the
  Peng et al. normal-tissue reference cannot match, rather than a fixable pipeline artifact.


### Step 1: the paper's reference-processing procedure, and a data limitation

From XDec-CHI's STAR Methods ("Processing RNA-seq datasets" / the preceding methodology paragraph):

> "PDAC single cell RNA-seq profiles were downloaded from GSA: CRA001160 (Peng et al., 2019a). Metadata from
> the study gave cell type identity for each cell. Profiles with low coverage (<300 genes/cell) and cells
> with coverage in the bottom quartile were removed. For each cell type, cells were ranked by total coverage
> and then pseudo-bulk profiles were created by summing every 5 profiles. Pseudo-bulk depth was then
> normalized to match the sample with the highest coverage and transformed into a 0-1 range using Equation 1.
> 100 pseudo-bulks were created for all cell types resulting in 11 cell type references."

Equation 1: `y = 1 - e^(-ax); a = 1/max{x}` (applied per gene, across all samples).

**Data limitation found while implementing this**: this project's exported reference (`refs.tsv`, from the
Besca-reprocessed `Zenodo 3969339` h5ad) has **already been depth-normalized to exactly 10,000 counts/cell**
by Besca's own upstream pipeline, before this project ever downloaded it (every single cell's total sums to
10,000 +/- floating-point noise - verified directly: `refs.sum(axis=0)` has std=0.0006 across all 2,748
cells). The paper's "<300 genes/cell" filter removes 0 cells here (already satisfied), but its
"bottom-quartile-of-coverage" filter and "rank cells by total coverage" step **cannot be meaningfully
replicated** - the real sequencing-depth information those steps need was discarded upstream, before this
project has access to it. What follows implements everything that *is* replicable (pseudo-bulk summing for
noise reduction, depth-matching, the Equation 1 transform) and uses a fixed random grouping of 5 cells per
pseudo-bulk instead of a coverage rank that doesn't meaningfully exist in this source data - documented here
rather than silently assumed.

In [1]:
import os
import re

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

# This notebook lives in progress/paper_duplication/; Jupyter starts the kernel with
# cwd = the notebook's own folder, and every "data/..." path below is relative to progress/.
if os.path.basename(os.getcwd()) == "paper_duplication":
    os.chdir("..")

XDEC_DIR = "data/deconvolution/xdec"
PAPER_REF_DIR = os.path.join(XDEC_DIR, "paper_style_ref")
os.makedirs(PAPER_REF_DIR, exist_ok=True)

PYTHON_PIXI = os.path.expanduser("~/.pixi/envs/python/bin/python3")
XDEC_SCRIPT = "tools/XDec_SM/python/xdec.py"

restricted_bulk_path = os.path.join(XDEC_DIR, "TCGA-PAAD_counts_raw_n150.tsv")
paper_genes_path = os.path.join(XDEC_DIR, "gene_lists", "xdec_chi_table_s3_genes.txt")

import subprocess
os.getcwd()


'/Users/liruixi/Desktop/i20_transfer/QCB/rotation/Alex/XDec_Pancreatic_Cancer/progress'

### Step 2: re-export up to 500 single cells/class

Notebook 04's original export capped at 300 cells/class - not enough to reach the paper's 100-pseudo-bulks-
per-type target after summing groups of 5 (300/5 = 60 max). Re-exporting from the raw h5ad with a 500-cell
cap allows the 9 major classes to reach the full 100; `neural cell` has only 48 cells total in the whole
dataset, so it is capped there regardless (9 pseudo-bulks) - the paper reports 11 cell type references
(including a removed "Endothelial 1" split this project's single `blood vessel endothelial cell` label has
no analog for), so an exact 1:1 match in resulting pseudo-bulk counts per type isn't expected either way.

In [2]:
refs_500_path = os.path.join(PAPER_REF_DIR, "refs_500.tsv")
meta_500_path = os.path.join(PAPER_REF_DIR, "ref_metadata_500.tsv")

if not os.path.exists(refs_500_path):
    import scanpy as sc

    adata = sc.read_h5ad("data/raw/zenodo_3969339/StdWf1_PRJCA001063_annotated.h5ad")
    rng = np.random.default_rng(1)
    class_col = "celltype1"
    max_cells_per_class = 500

    classes = adata.obs[class_col].astype(str)
    keep_names = []
    for cls, names in classes.groupby(classes).groups.items():
        names = np.array(names)
        n = min(max_cells_per_class, len(names))
        keep_names.extend(rng.choice(names, size=n, replace=False))

    sub_raw = adata.raw[keep_names]
    counts_like = np.expm1(sub_raw.X)
    try:
        counts_like = counts_like.toarray()
    except AttributeError:
        pass
    counts_like = np.asarray(counts_like)
    counts_like[counts_like < 0] = 0.0

    cells_idx = pd.Index(keep_names)
    refs_500 = pd.DataFrame(counts_like.T, index=sub_raw.var_names, columns=cells_idx)
    refs_500.index.name = "Genes"
    refs_500.to_csv(refs_500_path, sep="\t")

    meta_500 = pd.DataFrame({"Sample_ID": cells_idx, "Cell_Type": classes.loc[cells_idx].values})
    meta_500.to_csv(meta_500_path, sep="\t", index=False)

refs_500 = pd.read_csv(refs_500_path, sep="\t", index_col=0)
ref_meta_500 = pd.read_csv(meta_500_path, sep="\t", index_col=0)
print(refs_500.shape)
print(ref_meta_500["Cell_Type"].value_counts())


(17004, 4548)
Cell_Type
T cell                           500
blood vessel endothelial cell    500
enteroendocrine cell             500
fibroblast                       500
lymphocyte of B lineage          500
myeloid leukocyte                500
pancreatic acinar cell           500
pancreatic ductal cell           500
pancreatic stellate cell         500
neural cell                       48
Name: count, dtype: int64


### Step 3-4: pseudo-bulk (sum of 5), depth-match, Equation 1 transform

In [3]:
refs_paperstyle_path = os.path.join(PAPER_REF_DIR, "refs_paperstyle.tsv")
meta_paperstyle_path = os.path.join(PAPER_REF_DIR, "ref_metadata_paperstyle.tsv")

if not os.path.exists(refs_paperstyle_path):
    cell_types_500 = ref_meta_500.loc[refs_500.columns, "Cell_Type"]
    MAX_PSEUDOBULKS_PER_TYPE = 100
    GROUP_SIZE = 5
    rng = np.random.default_rng(1)

    pseudobulk_cols = {}
    meta_rows = []
    for ct in sorted(cell_types_500.unique()):
        cells_ct = cell_types_500[cell_types_500 == ct].index.to_numpy()
        order = rng.permutation(cells_ct)  # no real coverage to rank by - see Step 1 note
        n_groups = min(MAX_PSEUDOBULKS_PER_TYPE, len(order) // GROUP_SIZE)
        for i in range(n_groups):
            grp = order[i*GROUP_SIZE:(i+1)*GROUP_SIZE]
            pb_id = f"{ct.replace(' ', '_')}_pb{i+1}"
            pseudobulk_cols[pb_id] = refs_500[grp].sum(axis=1)
            meta_rows.append({"Sample_ID": pb_id, "Cell_Type": ct})

    pseudobulk = pd.DataFrame(pseudobulk_cols)
    pb_meta = pd.DataFrame(meta_rows)
    print("pseudobulk shape:", pseudobulk.shape)
    print(pb_meta["Cell_Type"].value_counts().to_string())

    # Depth-match: scale every pseudo-bulk up to the depth of the highest-depth one.
    depth = pseudobulk.sum(axis=0)
    scale = depth.max() / depth
    pseudobulk_dm = pseudobulk.mul(scale, axis=1)

    # Equation 1: y = 1 - exp(-a*x), a = 1/max(x), per gene across all pseudo-bulks.
    gene_max = pseudobulk_dm.max(axis=1).replace(0, np.nan)
    a = 1.0 / gene_max
    transformed = (1 - np.exp(-pseudobulk_dm.mul(a, axis=0))).fillna(0.0)
    transformed.index.name = "Genes"

    transformed.to_csv(refs_paperstyle_path, sep="\t")
    pb_meta.to_csv(meta_paperstyle_path, sep="\t", index=False)

refs_paperstyle = pd.read_csv(refs_paperstyle_path, sep="\t", index_col=0)
print("paper-style reference:", refs_paperstyle.shape,
      "value range:", refs_paperstyle.values.min(), "-", round(refs_paperstyle.values.max(), 3))


paper-style reference: (17004, 909) value range: 0.0 - 0.632


### Step 5: rerun XDec Stage 1/2 and re-apply the three artifact signals

**Caveat surfaced by this run, not fixed here**: the paper's own pipeline also puts the *bulk* tumor matrix
through the same Equation 1 transform before Stage 1, bounding it to the same 0-1 scale as the reference.
This notebook only changed the reference's processing (per the specific question this notebook tests) - the
bulk matrix here is still raw counts (`TCGA-PAAD_counts_raw.tsv`, values up to the millions), so
`xdec.py`'s own joint quantile-normalization step (triggered when `--refs` is passed) now has to reconcile a
0-0.63-bounded reference against an unbounded raw-count bulk matrix - a substantially *larger* scale
mismatch than the original reference (which was on a roughly comparable raw-count-like scale to the bulk).
This is flagged as an alternative explanation for any result below, not ruled out.

In [4]:
ribo_hk_pattern = re.compile(r"^(RPL|RPS|MRPL|MRPS|EEF|RPLP|TMSB|B2M$|ACTB$|GAPDH$|TUBB|TUBA|HSP|FTL$|FTH1$)")

stage1_dir = os.path.join(XDEC_DIR, "stage1_with_refs_k9_n150_paperGenes_paperRef")
stage2_dir = os.path.join(XDEC_DIR, "stage2_with_refs_k9_n150_paperGenes_paperRef")
os.makedirs(stage1_dir, exist_ok=True)
os.makedirs(stage2_dir, exist_ok=True)
prop_path = os.path.join(stage1_dir, "proportions.tsv")
means_path = os.path.join(stage2_dir, "stage2_means.tsv")

if not os.path.exists(prop_path):
    result = subprocess.run(
        [PYTHON_PIXI, XDEC_SCRIPT, "stage1",
         "--bulk", restricted_bulk_path,
         "--probes", paper_genes_path,
         "--refs", refs_paperstyle_path,
         "--ref-metadata", meta_paperstyle_path,
         "--k", "9",
         "--outdir", stage1_dir],
        capture_output=True, text=True,
    )
    print("stage1:", result.returncode)
    if result.returncode != 0:
        print(result.stdout, result.stderr)

if not os.path.exists(means_path):
    result = subprocess.run(
        [PYTHON_PIXI, XDEC_SCRIPT, "stage2",
         "--bulk-counts", restricted_bulk_path,
         "--proportions", prop_path,
         "--outdir", stage2_dir],
        capture_output=True, text=True,
    )
    print("stage2:", result.returncode)
    if result.returncode != 0:
        print(result.stdout, result.stderr)

print("done")


done


In [5]:
def diagnose(label, s1_dir, s2_dir, n_probes):
    props_r = pd.read_csv(os.path.join(s1_dir, "proportions.tsv"), sep="\t", index_col=0)
    corr_r = pd.read_csv(os.path.join(s1_dir, "correlation.tsv"), sep="\t", index_col=0)
    means_r = pd.read_csv(os.path.join(s2_dir, "stage2_means.tsv"), sep="\t", index_col=0)

    dom = props_r.idxmax(axis=1).value_counts()
    breadth = (corr_r > 0.3).sum(axis=0).sort_values(ascending=False)
    candidate = breadth.idxmax()

    cand_means = means_r[candidate]
    zf_rows = []
    for other in means_r.columns:
        if other == candidate:
            continue
        mask = (means_r[other] == 0) & (cand_means > cand_means.quantile(0.99))
        zf_rows.append((other, int(mask.sum())))
    zf_df = pd.DataFrame(zf_rows, columns=["partner", "n"]).sort_values("n", ascending=False)
    best_partner = zf_df.iloc[0]["partner"]
    top_genes = cand_means[(means_r[best_partner] == 0) & (cand_means > cand_means.quantile(0.99))] \
        .sort_values(ascending=False).head(20)
    frac_hk = top_genes.index.str.match(ribo_hk_pattern).mean()

    most_dominant_profile = dom.idxmax()
    most_dominant_n = int(dom.max())
    most_dominant_breadth = int(breadth.get(most_dominant_profile, 0))

    return {
        "run": label, "n_probes": n_probes,
        "broadest_profile": candidate, "max_breadth": int(breadth.max()),
        "broadest_profile_dominant_n": int(dom.get(candidate, 0)),
        "most_dominant_profile": most_dominant_profile, "most_dominant_n": most_dominant_n,
        "most_dominant_profile_breadth": most_dominant_breadth,
        "zero_floor_partner": best_partner, "n_zero_floor_genes": int(zf_df.iloc[0]["n"]),
        "frac_top20_housekeeping": round(frac_hk, 2),
    }

rows = []
rows.append(diagnose("baseline (344 genes, original ref)",
                      os.path.join(XDEC_DIR, "stage1_with_refs_k9_n150"),
                      os.path.join(XDEC_DIR, "stage2_with_refs_k9_n150"), 344))
rows.append(diagnose("A: paper genes, original ref",
                      os.path.join(XDEC_DIR, "stage1_with_refs_k9_n150_paperGenes"),
                      os.path.join(XDEC_DIR, "stage2_with_refs_k9_n150_paperGenes"), 282))
rows.append(diagnose("B: no housekeeping, original ref",
                      os.path.join(XDEC_DIR, "stage1_with_refs_k9_n150_noHK"),
                      os.path.join(XDEC_DIR, "stage2_with_refs_k9_n150_noHK"), 292))
rows.append(diagnose("C: paper genes, paper-style ref (this notebook)", stage1_dir, stage2_dir, 282))

results_df = pd.DataFrame(rows).set_index("run")
results_df


,n_probes,broadest_profile,max_breadth,broadest_profile_dominant_n,most_dominant_profile,most_dominant_n,most_dominant_profile_breadth,zero_floor_partner,n_zero_floor_genes,frac_top20_housekeeping
run,,,,,,,,,,
"baseline (344 genes, original ref)",344,Profile_7,10,4,Profile_1,88,0,Profile_2,118,0.15
"A: paper genes, original ref",282,Profile_8,5,0,Profile_9,88,0,Profile_7,163,0.65
"B: no housekeeping, original ref",292,Profile_8,4,21,Profile_7,82,0,Profile_5,196,0.65
"C: paper genes, paper-style ref (this notebook)",282,Profile_1,6,0,Profile_6,87,0,Profile_4,166,0.60


In [6]:
print("Full correlation matrix, Run C (paper genes, paper-style reference):")
corr_c = pd.read_csv(os.path.join(stage1_dir, "correlation.tsv"), sep="\t", index_col=0)
corr_c.round(3)


Full correlation matrix, Run C (paper genes, paper-style reference):


,Profile_1,Profile_2,Profile_3,Profile_4,Profile_5,Profile_6,Profile_7,Profile_8,Profile_9
T cell,0.414,0.222,-0.267,-0.269,0.149,-0.243,0.131,0.089,0.009
blood vessel endothelial cell,0.391,-0.118,-0.198,-0.137,0.278,-0.212,0.209,-0.041,0.088
enteroendocrine cell,0.307,-0.237,-0.085,0.248,0.116,-0.128,0.170,-0.136,-0.214
fibroblast,0.236,-0.109,-0.352,-0.216,0.389,-0.170,-0.015,-0.075,0.455
lymphocyte of B lineage,0.404,0.264,-0.258,-0.273,0.143,-0.241,0.103,0.057,0.042
myeloid leukocyte,0.258,0.224,-0.204,-0.251,0.331,-0.372,0.039,0.222,0.058
neural cell,0.369,0.155,-0.272,-0.239,0.211,-0.231,0.132,0.086,0.076
pancreatic acinar cell,0.280,-0.246,0.247,-0.134,-0.027,-0.177,0.255,-0.106,-0.230
pancreatic ductal cell,0.363,-0.310,-0.117,-0.145,0.160,-0.126,0.523,-0.031,-0.216
pancreatic stellate cell,0.274,-0.020,-0.248,-0.135,0.295,-0.241,0.060,-0.032,0.376


### Step 6: a real bug in Run C, and the corrected test

**Run C's own setup was flawed, not just caveated.** Reading `tools/XDec_SM/python/xdec.py` directly reveals
that it is a near-literal Python port of the paper's own R codebase - its docstrings cite
`XDec_total_code_vpetrosyan.Rmd` (Varduhi Petrosyan is a co-author of XDec-CHI). Its normalization pipeline,
run automatically whenever `--refs` is passed (`normalise_bulk()`, `normalise_refs_independent()`), **already
implements exactly the paper's procedure**: joint quantile normalization of bulk + reference, then
`logistic_a100()` - which is *literally* Equation 1 (`y = 1 - exp(-a*x), a = 1/max(x)`) - applied per gene.

Run C fed a reference that **this project had already manually transformed** with Equation 1
(`refs_paperstyle.tsv`, bounded 0-0.63) into `--refs`. `xdec.py` then applied its own joint quantile
normalization + `logistic_a100` **on top of that** - a second, redundant application of the same transform,
which also means the joint quantile-normalization step that `bulk_norm` is computed from was matched against
an already-squashed reference distribution rather than a raw one. Run C's "no improvement" result is
therefore confounded by this bug, not a clean negative result.

**Corrected test**: build the pseudo-bulk reference as **raw, untransformed** summed counts (same grouping as
Step 3, skipping Step 4's manual depth-matching and Equation 1 entirely) and let `xdec.py`'s own pipeline -
already correct - do all normalization, exactly as every other run in this project has relied on it to do.

In [7]:
refs_pbraw_path = os.path.join(PAPER_REF_DIR, "refs_pseudobulk_raw.tsv")
meta_pbraw_path = os.path.join(PAPER_REF_DIR, "ref_metadata_pseudobulk_raw.tsv")

if not os.path.exists(refs_pbraw_path):
    cell_types_500 = ref_meta_500.loc[refs_500.columns, "Cell_Type"]
    MAX_PSEUDOBULKS_PER_TYPE = 100
    GROUP_SIZE = 5
    rng = np.random.default_rng(1)

    pseudobulk_cols = {}
    meta_rows = []
    for ct in sorted(cell_types_500.unique()):
        cells_ct = cell_types_500[cell_types_500 == ct].index.to_numpy()
        order = rng.permutation(cells_ct)
        n_groups = min(MAX_PSEUDOBULKS_PER_TYPE, len(order) // GROUP_SIZE)
        for i in range(n_groups):
            grp = order[i*GROUP_SIZE:(i+1)*GROUP_SIZE]
            pb_id = f"{ct.replace(' ', '_')}_pb{i+1}"
            pseudobulk_cols[pb_id] = refs_500[grp].sum(axis=1)
            meta_rows.append({"Sample_ID": pb_id, "Cell_Type": ct})

    pseudobulk_raw = pd.DataFrame(pseudobulk_cols)
    pseudobulk_raw.index.name = "Genes"
    pb_meta_raw = pd.DataFrame(meta_rows)
    pseudobulk_raw.to_csv(refs_pbraw_path, sep="\t")
    pb_meta_raw.to_csv(meta_pbraw_path, sep="\t", index=False)

refs_pbraw = pd.read_csv(refs_pbraw_path, sep="\t", index_col=0)
print("raw pseudo-bulk reference (no manual transform):", refs_pbraw.shape,
      "value range:", refs_pbraw.values.min(), "-", round(refs_pbraw.values.max(), 1))


raw pseudo-bulk reference (no manual transform): (17004, 909) value range: 0.0 - 25166.8


In [8]:
stage1_dir_d = os.path.join(XDEC_DIR, "stage1_with_refs_k9_n150_paperGenes_pseudobulkRaw")
stage2_dir_d = os.path.join(XDEC_DIR, "stage2_with_refs_k9_n150_paperGenes_pseudobulkRaw")
os.makedirs(stage1_dir_d, exist_ok=True)
os.makedirs(stage2_dir_d, exist_ok=True)
prop_path_d = os.path.join(stage1_dir_d, "proportions.tsv")
means_path_d = os.path.join(stage2_dir_d, "stage2_means.tsv")

if not os.path.exists(prop_path_d):
    result = subprocess.run(
        [PYTHON_PIXI, XDEC_SCRIPT, "stage1",
         "--bulk", restricted_bulk_path,
         "--probes", paper_genes_path,
         "--refs", refs_pbraw_path,
         "--ref-metadata", meta_pbraw_path,
         "--k", "9",
         "--outdir", stage1_dir_d],
        capture_output=True, text=True,
    )
    print("stage1:", result.returncode)
    if result.returncode != 0:
        print(result.stdout, result.stderr)

if not os.path.exists(means_path_d):
    result = subprocess.run(
        [PYTHON_PIXI, XDEC_SCRIPT, "stage2",
         "--bulk-counts", restricted_bulk_path,
         "--proportions", prop_path_d,
         "--outdir", stage2_dir_d],
        capture_output=True, text=True,
    )
    print("stage2:", result.returncode)
    if result.returncode != 0:
        print(result.stdout, result.stderr)

print("done")


done


In [9]:
rows.append(diagnose("D: paper genes, RAW pseudo-bulk ref (corrected - xdec.py normalizes)",
                      stage1_dir_d, stage2_dir_d, 282))
results_df = pd.DataFrame(rows).set_index("run")
results_df


,n_probes,broadest_profile,max_breadth,broadest_profile_dominant_n,most_dominant_profile,most_dominant_n,most_dominant_profile_breadth,zero_floor_partner,n_zero_floor_genes,frac_top20_housekeeping
run,,,,,,,,,,
"baseline (344 genes, original ref)",344,Profile_7,10,4,Profile_1,88,0,Profile_2,118,0.15
"A: paper genes, original ref",282,Profile_8,5,0,Profile_9,88,0,Profile_7,163,0.65
"B: no housekeeping, original ref",292,Profile_8,4,21,Profile_7,82,0,Profile_5,196,0.65
"C: paper genes, paper-style ref (this notebook)",282,Profile_1,6,0,Profile_6,87,0,Profile_4,166,0.60
"D: paper genes, RAW pseudo-bulk ref (corrected - xdec.py normalizes)",282,Profile_1,7,0,Profile_4,102,0,Profile_5,156,0.60


In [10]:
print("Full correlation matrix, Run D (paper genes, RAW pseudo-bulk reference):")
corr_d = pd.read_csv(os.path.join(stage1_dir_d, "correlation.tsv"), sep="\t", index_col=0)
corr_d.round(3)


Full correlation matrix, Run D (paper genes, RAW pseudo-bulk reference):


,Profile_1,Profile_2,Profile_3,Profile_4,Profile_5,Profile_6,Profile_7,Profile_8,Profile_9
T cell,0.449,0.203,0.113,-0.165,-0.243,-0.318,0.163,-0.066,0.184
blood vessel endothelial cell,0.398,-0.123,0.217,-0.144,-0.108,-0.210,0.010,0.042,0.289
enteroendocrine cell,0.247,-0.232,0.187,-0.092,0.283,-0.035,-0.112,-0.165,0.141
fibroblast,0.300,-0.088,0.016,-0.161,-0.183,-0.361,0.017,0.401,0.401
lymphocyte of B lineage,0.440,0.242,0.078,-0.173,-0.249,-0.323,0.129,-0.036,0.174
myeloid leukocyte,0.292,0.198,0.036,-0.293,-0.227,-0.246,0.272,-0.027,0.349
neural cell,0.409,0.137,0.121,-0.160,-0.200,-0.320,0.164,-0.005,0.249
pancreatic acinar cell,0.190,-0.340,0.238,-0.150,-0.169,0.227,-0.169,-0.303,-0.072
pancreatic ductal cell,0.379,-0.271,0.518,0.051,-0.095,-0.132,0.014,-0.222,0.178
pancreatic stellate cell,0.329,-0.008,0.090,-0.213,-0.112,-0.263,0.046,0.321,0.306


**Interpretation of the corrected test (Run D)**

With the double-transformation bug fixed, pseudo-bulking is **still not the missing piece - if anything, both
recurring patterns get slightly more extreme**, not better:

* **Broadest-correlating profile**: breadth=7/10 (Profile_1) - higher than every prior run (baseline's 10 is
  still the ceiling, but 7 is worse than A's 5, B's 4, and even the buggy Run C's 6).
* **Dominant-majority, zero-correlation profile**: Profile_4, dominant in **102/150 samples (68%)** - the
  largest of any run tested (previous range was 82-88/150, ~55-59%), still with correlation breadth exactly
  0 against all 10 reference types.
* Zero-floor housekeeping fraction (0.60) is unchanged from the buggy Run C, consistent with Step-6/
  notebook-01's finding that this signal tracks the full 53,885-gene Stage-2 refit, not the reference used to
  drive Stage 1.

**This is now a clean, methodologically sound negative result** (not confounded by double-transformation):
single-cell vs. pseudo-bulk reference processing, with `xdec.py`'s own paper-matching normalization pipeline
left untouched in both cases, does not explain either recurring pattern. Combined with Steps 5-6's gene-list
test, none of the three candidate explanations tested in this project (cohort size, gene list, reference
processing) - alone or in combination - eliminates the dominant-majority/zero-correlation profile. Its
remarkable consistency in magnitude (now spanning 82-102/150, ~55-68%, always breadth exactly 0) across every
single configuration tried strengthens, rather than weakens, the hypothesis raised in notebook 01's Summary:
this looks like real tumor-epithelial signal that the Peng et al. normal-tissue reference cannot match,
not an estimation defect to be engineered away.